# Creating FHIR resources with Python

This notebook creates FHIR resources in Python with `fhir.resources`, sends them to the FHIR server, and shows how the server responds to an invalid resource. It assumes a FHIR server with data already loaded, as set up in [Loading FHIR data](3.1-load-fhir-data.ipynb).


## Creating new FHIR data 

Let's say we want to create a new resource regarding some kind of medical data, rather than just use synthetic generated data. Let's do this using Python.

We are going to add FHIR data to the server in two steps, firstly create a valid fhir resource - the Python package [`fhir.resources`](https://github.com/nazrulworld/fhir.resources) is going to help with this. Then sending a POST request to the server using the standard http `requests` library. 

### Step 1: Create valid resources

Firstly, install the dependency if you haven't already (from requirements.txt). This is [fhir.resources](https://github.com/nazrulworld/fhir.resources). There is much more information about its usage on the linked repo than included here, so check it out if you want more information.


In [1]:
# pip install fhir.resources 

You then need to import the type of resource you would like to make. We will start by creating a new patient using a JSON format object:

In [2]:
from fhir.resources.patient import Patient

data = { ## Create a data object that resembles a fhir resource
    "name": [{
        "use": "official",
        "family": "Kent", 
        "given": ["Clark"]  ## As you can have multiple given names (middle names) this needs to be a list item
    }],
    "birthDate":"1965-02-12", "gender":"male"}
superman = Patient.model_validate(data)

In [3]:
print(superman.birthDate)
print(superman.name)

1965-02-12
[HumanName(fhir_comments=None, extension=None, id=None, family='Kent', family__ext=None, given=['Clark'], given__ext=None, period=None, prefix=None, prefix__ext=None, suffix=None, suffix__ext=None, text=None, text__ext=None, use='official', use__ext=None)]


Alternatively we can use a more Pythonic type way:

In [4]:
batman = Patient.model_construct()


batman.name = [{}] # The name part of the patient resource is a bit of a pain! 
batman.id = "500"
batman.name[0].family = "Wayne"
batman.name[0].given = ["Bruce"]
batman.birthDate = "1982-01-15"
batman.gender= "male"

#### Accessing model data

In [5]:
batman_model = batman.model_dump_json() 
print(batman_model,'\n' ,  type(batman_model), '\n\n') ## This returns a JSON string

superman_model = superman.model_dump() ## This returns a json object (or python dict)

print(superman_model, '\n', type(superman_model))

{"resourceType":"Patient","id":"500","name":[{"family":"Wayne","given":["Bruce"]}],"gender":"male","birthDate":"1982-01-15"} 
 <class 'str'> 


OrderedDict({'resourceType': 'Patient', 'name': [OrderedDict({'use': 'official', 'family': 'Kent', 'given': ['Clark']})], 'gender': 'male', 'birthDate': datetime.date(1965, 2, 12)}) 
 <class 'collections.OrderedDict'>


There are classes for every fhir resource defined in the FHIR specification, so a similar method can be used with any resource you may need.

Below we can create a DocumentReference resource. If you have worked through the main demo in this repository, you may be familiar with this resource type, but it can be used to link any different type of documents. 


In [6]:
from fhir.resources.documentreference import DocumentReference
from datetime import datetime, timezone
note_resource = DocumentReference.model_construct()


with open("../data/clinical_note.txt", "r", encoding="utf-8") as f:
    note = f.read()
    
# Set the note status
note_resource.status = "current"

# Set the date
note_resource.date = datetime(2026, 9, 10, tzinfo=timezone.utc)
# Set the subject reference (patient ID)
note_resource.subject = {"reference": "500"}
# Set the content with the note text as an attachment, encoding the content in hexadecimal
note_resource.content = [{ "attachment":{"contentType": "text/plain; charset=utf-8", "data": note.encode("utf-8").hex()}}]


### Step 2: Adding data to the server

Similar to how we accessed the data with http requests in the [Querying FHIR data with Python](3.2-querying-fhir-data-with-python.ipynb) demo, we can add the FHIR resources we have created to the server using HTTP requests. We do need to add basic authorisation. 

In [7]:
import requests
from requests.auth import HTTPBasicAuth

## Credentials
username = "SuperUser"
password = "SYS"

## FHIR server location
baseURL = "http://localhost:52773/fhir/r4/"
headers ={    "Content-Type": "application/fhir+json"}
endpoint = "Patient"


res = requests.post(baseURL+endpoint, 
                    data=superman.model_dump_json(), ## Give a JSON string as data - giving it a Python dict may cause errors because of the datetime object.
                    headers= headers, ## We are defining the content type (FHIR+JSON) in the header
                    auth=HTTPBasicAuth(username, password))
print(res)

# The server assigns the ID; it is the part of the Location header after Patient/
patient_id = res.headers["Location"].split("Patient/")[1].split("/")[0]
print(patient_id)

<Response [201]>
2287


In [8]:
## If you have any error message, you can find some more information with 
if res.status_code!=201:
    print(res.json())
else: 
    print("Data successfully added!")


Data successfully added!


Response 201 means it has been successfully added!

In [9]:
res = requests.post(baseURL+endpoint, data= batman.model_dump_json(), headers= headers, auth=HTTPBasicAuth(username, password))

print(res)

<Response [201]>


#### Checking data is added
Lets just query the endpoint to see whether we have added them correctly:

In [10]:
## FHIR server location
baseURL = "http://localhost:52773/fhir/r4/"

get_headers ={    "Accept": "application/fhir+json"}

endpoint = "Patient"


res = requests.get(baseURL+endpoint, headers=get_headers, auth=HTTPBasicAuth(username, password))
print(res)
#print(res.json())
data = res.json() 

for entry in data.get("entry", [])[-5:]: ## Get the last 5 entries
    patient = entry.get('resource', {}) ## Get the patient resource
    print(patient.get('id'), patient.get('name'), patient.get('gender')) ## Print the patient info


<Response [200]>


885 [{'use': 'official', 'family': 'Schowalter414', 'given': ['Kristofer887'], 'prefix': ['Mr.']}] male
1068 [{'use': 'official', 'family': 'Mosciski958', 'given': ['Marshall526'], 'prefix': ['Mr.']}] male
1741 [{'use': 'official', 'family': 'Gaylord332', 'given': ['Milissa240'], 'prefix': ['Ms.']}] female
2287 [{'use': 'official', 'family': 'Kent', 'given': ['Clark']}] male
2288 [{'family': 'Wayne', 'given': ['Bruce']}] male


So we have added them! Because we have used a POST request, they are automatically assigned an ID number. We could use a put request if we want to keep the ID number we've assigned them.


### Adding Document Reference

Lets do the same with the document reference resource - only thing we need to change is the endpoint and the data:

In [11]:
res = requests.post(baseURL+"DocumentReference", data=note_resource.model_dump_json(),
                   headers={"Content-Type": "application/fhir+json"}, auth=HTTPBasicAuth(username, password))

print(res)
print(res.json())

<Response [400]>
{'resourceType': 'OperationOutcome', 'issue': [{'severity': 'error', 'code': 'invalid', 'diagnostics': '<HSFHIRErr>MalformedRelativeReference', 'details': {'text': "The reference value '500' in property (subject) of Type 'DocumentReference' is malformed"}, 'expression': ['DocumentReference.subject']}]}


Ok, so the fhir.resources doesn't completely validate our data, but the FHIR server rejects bad data! Lets work through these issues: 

In [12]:
for issue in res.json()["issue"]:
    print(issue["details"]["text"])
    print('\n')

The reference value '500' in property (subject) of Type 'DocumentReference' is malformed




We can see the problem here - the patient reference is wrong. Lets fix it and try again: 

In [13]:
note_resource.subject = {"reference": f"Patient/{patient_id}"}


In [14]:
res = requests.post(baseURL+"DocumentReference", data=note_resource.model_dump_json(),
                   headers={"Content-Type": "application/fhir+json"}, auth=HTTPBasicAuth(username, password))

print(res)


<Response [201]>


In [15]:
res = requests.get(baseURL+f"DocumentReference?patient={patient_id}",
                   headers=get_headers, auth=HTTPBasicAuth(username, password))
print(res)

<Response [200]>


In [16]:
resource = res.json()["entry"][0]["resource"]
# print(resource)
print(bytes.fromhex(resource["content"][0]["attachment"]["data"]).decode("utf-8"))

Patient Name: Clark Kent
DOB: [Redacted]
Date of Visit: 09/10/2025
Presenting Complaint:
Patient presents with complaints of persistent chest discomfort and intermittent episodes of spontaneous x-ray vision activation, particularly during emotionally charged situations (e.g., office meetings, romantic encounters, surprise birthday parties).
History of Present Illness:
Mr. Kent reports a recent increase in involuntary activation of his x-ray vision, resulting in several awkward social encounters, including accidentally seeing through three floors of the Daily Planet building. He also notes mild chest tightness after flying through a cumulonimbus cloud at Mach 3, which he describes as “like being hugged by a thunderstorm.”
Past Medical History:

Kryptonian physiology
Solar overexposure syndrome (resolved)
Mild allergic reaction to kryptonite (ongoing)

Medications:

None (immune to most Earth-based pharmaceuticals)

Allergies:

Kryptonite (severe photosensitivity, weakness, existential d